In [1]:
import pandas as pd
import torch
import os
import numpy as np

In [3]:
from datasets import Dataset
from transformers import BertTokenizer, TrainingArguments, Trainer

c:\Users\User\Documents\devanasokan_fyp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df = pd.read_csv("C:/Users/User/Documents/devanasokan_fyp/02_LLMLabelling/labeled_songs.csv")

In [5]:
# Select only the columns you need
# 'label' must be 0 or 1
df_bert = df[['lyrics', 'label']]

# Convert to Hugging Face format
dataset = Dataset.from_pandas(df_bert)

In [7]:
# Fixes the [WinError 3] by explicitly setting a local cache directory
cache_dir = "C:/Users/User/Documents/devanasokan_fyp/huggingface_cache"
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)

In [8]:
# Tell Hugging Face to use this directory
os.environ['HF_HOME'] = cache_dir

# This turns off the annoying symlink warning
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

In [9]:
# Initiate with the cache path
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', cache_dir=cache_dir)

c:\Users\User\Documents\devanasokan_fyp\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\Documents\devanasokan_fyp\huggingface_cache\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [10]:
def preprocess_function(examples):
    # This creates the 'input_ids' and 'attention_mask' BERT needs
    return tokenizer(examples["lyrics"], truncation=True, padding="max_length", max_length=512)

tokenized_dataset = dataset.map(preprocess_function, batched=True)

Map: 100%|██████████| 4385/4385 [00:02<00:00, 1958.16 examples/s]


In [11]:
# 80% Train, 20% Test
full_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)

In [12]:
print(full_dataset) 
# If it shows {'train': ..., 'test': ...}, it is already split!

DatasetDict({
    train: Dataset({
        features: ['lyrics', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3508
    })
    test: Dataset({
        features: ['lyrics', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 877
    })
})


In [13]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "distilbert-base-uncased" # Or any model from the Hugging Face Hub

# 1. Load the tokenizer (must match the model)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Load the model with a classification head
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 364.56it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
import evaluate
metric = evaluate.load("accuracy")

In [15]:
print(metric)

EvaluationModule(name: "accuracy", module_type: "metric", features: {'predictions': Value('int32'), 'references': Value('int32')}, usage: """
Args:
    predictions (`list` of `int`): Predicted labels.
    references (`list` of `int`): Ground truth labels.
    normalize (`boolean`): If set to False, returns the number of correctly classified samples. Otherwise, returns the fraction of correctly classified samples. Defaults to True.
    sample_weight (`list` of `float`): Sample weights Defaults to None.

Returns:
    accuracy (`float` or `int`): Accuracy score. Minimum possible value is 0. Maximum possible value is 1.0, or the number of examples input, if `normalize` is set to `True`.. A higher score means higher accuracy.

Examples:

    Example 1-A simple example
        >>> accuracy_metric = evaluate.load("accuracy")
        >>> results = accuracy_metric.compute(references=[0, 1, 2, 0, 1, 2], predictions=[0, 1, 1, 2, 1, 0])
        >>> print(results)
        {'accuracy': 0.5}

    Exa

In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # argmax picks the highest probability (0 or 1)
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [22]:
# Set training arguments
training_args = TrainingArguments(
    output_dir="./results",          # Folder where checkpoints are saved
    eval_strategy="epoch",     # Run evaluation after every epoch
    save_strategy="epoch",           # Save model after every epoch
    learning_rate=2e-5,              # Standard BERT fine-tuning rate
    per_device_train_batch_size=16,  # Batch size for training
    per_device_eval_batch_size=16,   # Batch size for evaluation
    num_train_epochs=1,              # Total passes through the data
    weight_decay=0.01,               # Regularization to prevent overfitting
    load_best_model_at_end=True,     # Keeps the best version of the model
    fp16=torch.cuda.is_available()  # Use Mixed Precision if on GPU for 2x speed
)

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`

In [28]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=full_dataset["train"],
    eval_dataset=full_dataset["test"],
    compute_metrics=compute_metrics,
)

print(trainer.compute_metrics)

<function compute_metrics at 0x00000235D87D0220>


In [29]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.098793,0.796078,0.826500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1000, training_loss=0.1185965690612793, metrics={'train_runtime': 427.2611, 'train_samples_per_second': 37.448, 'train_steps_per_second': 2.34, 'total_flos': 2119478378496000.0, 'train_loss': 0.1185965690612793, 'epoch': 1.0})

In [32]:
history = pd.DataFrame(trainer.state.log_history)
print(history)

       loss  grad_norm  learning_rate  epoch  step  eval_loss  eval_accuracy  \
0  0.138400   0.547643   1.006000e-05    0.5   500        NaN            NaN   
1  0.098793  34.950279   6.000000e-08    1.0  1000        NaN            NaN   
2       NaN        NaN            NaN    1.0  1000   0.796078         0.8265   
3       NaN        NaN            NaN    1.0  1000        NaN            NaN   
4       NaN        NaN            NaN    1.0  1000   0.796078         0.8265   
5       NaN        NaN            NaN    1.0  1000   0.796078         0.8265   

   eval_runtime  eval_samples_per_second  eval_steps_per_second  \
0           NaN                      NaN                    NaN   
1           NaN                      NaN                    NaN   
2       33.5213                  119.327                  7.458   
3           NaN                      NaN                    NaN   
4       33.2437                  120.323                  7.520   
5       37.8415                  105.

In [34]:
# Save the version currently in the trainer's brain
trainer.save_model("./my_final_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# TEST MODEL

from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

# Load the model and tokenizer from your local folder
path = "./my_final_model"
model = AutoModelForSequenceClassification.from_pretrained(path)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased") # Ensure you saved the tokenizer there too!

# Create a 'pipeline' (the easiest way to use the model)
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

# Test it on a new sentence
result = classifier("bad ending")
print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'LABEL_0', 'score': 0.9826032519340515}]


In [23]:
import accelerate
print(accelerate.__version__)

1.13.0


In [19]:
import accelerate
import transformers
print(f"Accelerate version: {accelerate.__version__}")
print(f"Transformers version: {transformers.__version__}")

Accelerate version: 1.13.0
Transformers version: 5.1.0


In [1]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.10.0+cpu
CUDA available: False
CUDA version: None


AssertionError: Torch not compiled with CUDA enabled